# Dock 06

## Configuration

In [1]:
# Environment configurations
# pull in local configuration
%run config.py
# display file to screen
!cat config.py

# config.py
# This file wants to be listed in .gitignore

GNINA_LOC = "/home/dwaine/octoberproject/gnina"
GNINA_PARAMETER = "--no_gpu"


In [45]:
# Non user configurations
resultsdir = "results/"
decoydir = "decoys/"
docked = resultsdir + "docked.sdf"
log = resultsdir + "gninalog.txt"
rmsdlog = resultsdir + "rmsd.txt"
results = resultsdir + "results.csv"
rawresults = resultsdir + "rawresults.csv"

In [3]:
# Docking configurations.
sort_by_cnnvs = False # Optionally sort by cnnvs. Our experiments did not support this as a useful option.

In [4]:
# Protein and Ligand folder names along with optional generated decoy filename
protein_dir_name = "protein-er-13"
ligand_dir_name = "erligand260"
#generated_decoy_file = "generated_decoys_activeER_filtered_FINAL.sdf"

In [5]:
sourceproteindir = "protein/" + protein_dir_name + "/processed/"
referenceliganddir = "protein/" + protein_dir_name + "/reference_ligand/"
sourceliganddir = "ligand/" + ligand_dir_name + "/processed/"
proteinlistfile = "protein/" + protein_dir_name + "/protein.json"
print (sourceproteindir)
print (referenceliganddir)
print (sourceliganddir)
print (proteinlistfile)

protein/protein-er-13/processed/
protein/protein-er-13/reference_ligand/
ligand/erligand260/processed/
protein/protein-er-13/protein.json


## Define generic functions

In [8]:
import copy
def reportdict(rows, columns):
    lines = []
    if len(rows) == 0:
        return (lines) 
    inner_keys = set()
    for r in rows.values():
        inner_keys.update(r.keys())
    col_widths = {}
    col_widths["id"] = max(len("id"), max(len(k) for k in rows))
    for col in inner_keys:
        header_len = len(col)
        data_len = max(len(str(r.get(col, ""))) for r in rows.values())
        col_widths[col] = max(header_len, data_len)
    header = "  ".join(f"{col:<{col_widths[col]}}" for col in columns)
    lines.append(header)
    for outer_key, inner in rows.items():
        cells = [f"{outer_key:<{col_widths['id']}}"]
        for col in columns[1:]:
            cells.append(f"{str(inner.get(col, '')):<{col_widths[col]}}")
        lines.append("  ".join(cells))
    return (lines)

In [ ]:
from openbabel import openbabel as ob
from rdkit import Chem

def writeonedecoytoFS(row):
    
    # Your RDKit mol
    rdkit_mol = row['ROMol']

    # Convert RDKit MolBlock → Open Babel OBMol
    molblock = Chem.MolToMolBlock(rdkit_mol)
    ob_mol = ob.OBMol()
    ob_conversion = ob.OBConversion()
    ob_conversion.SetInFormat("mdl")  # MOL block format
    ob_conversion.ReadString(ob_mol, molblock)

    # Write out GNINA-ready SDF
    ob_conversion.SetOutFormat("sdf")
    sdf_string = ob_conversion.WriteString(ob_mol)

    with open(decoydir + "decoyligand.sdf", "w") as f:
        f.write(sdf_string)

In [9]:
import time

def format_time(seconds):
    hours = int(seconds // 3600)
    minutes = int((seconds % 3600) // 60)
    secs = int(seconds % 60)
    return f"{hours:02d}:{minutes:02d}:{secs:02d}"

## Optional. Load and process decoy ligands from .sdf

In [ ]:
# optional block used to create decoy ligands.

import pandas as pd
from rdkit.Chem import PandasTools

decoydf = None
sdf_path = decoydir + generated_decoy_file

decoydf = PandasTools.LoadSDF(
    sdf_path,
    molColName="ROMol",
    smilesName="SMILES",
    includeFingerprints=False,
    removeHs=False,
    strictParsing=True
)

# Add numbered ID column
decoydf["ID"] = [f"decoy{i+1}" for i in range(len(decoydf))]

print(decoydf.head())
print(decoydf.columns)
print(decoydf.shape)     # (rows, columns)
print(decoydf.info(memory_usage='deep'))

## (Depricated) Populate docked ligand dictionary

In [ ]:
# Create empty ligand dictionary.
dockedliganddict = {}
# Required columns are:
# id = unique identifier
# localfilename = filename of .sdf including path relative to the location of dock.ipynb.

In [ ]:
# Iterate through contents of one or many directories and find .sdf files

from pathlib import Path

def find_sdf_files(directory):
    """Return a sorted list of .sdf files in directory."""
    return sorted(Path(directory).glob("*.sdf"))

sdf_files = find_sdf_files(referenceliganddir)
for file_path in sdf_files:
    id = filename_without_extension = Path(file_path).stem
    dockedliganddict[id] = {'id':id, 'localfilename': file_path}
    #print(file_path)

# Show content of dictionary
lines = reportdict(dockedliganddict, ["id","localfilename"])
print("\n".join(lines))

In [ ]:
# Optional. Provide a .json file of ligand files to populate the dictionary.

import json

# Read from text file
with open(ligdictjson, "r") as f:
    dockedliganddict = json.load(f)

# Which docked ligands are available?
lines = reportdict(dockedliganddict, ["id","source","prep","localfilename"])
print("\n".join(lines))

## Populate protein dictionary

In [11]:
# Obsolete.
# Iterate through contents of one or many directories and find .pdb files

proteindict = {}

from pathlib import Path

def find_sdf_files(directory):
    """Return a sorted list of .sdf files in directory."""
    return sorted(Path(directory).glob("*.pdb"))

sdf_files = find_sdf_files(sourceproteindir)
for file_path in sdf_files:
    id = filename_without_extension = Path(file_path).stem
    proteindict[id] = {'id':id, 'localfilename': file_path}

# Show content of dictionary
lines = reportdict(proteindict, ["id","localfilename","localfilename_fixed","ligandreferencefilename"])
print("\n".join(lines))

KeyError: 'localfilename_fixed'

In [13]:
# Read the protein.json file created by the acquire-protein-fix-protein notebook script.

proteindict = {}

import json

# Read from text file
with open(proteinlistfile, "r") as f:
    proteindict = json.load(f)    

lines = reportdict(proteindict, ["id","localfilename","localfilename_fixed","ligandreferencefilename"])
print("\n".join(lines))

id    localfilename  localfilename_fixed  ligandreferencefilename
1ERE  1ERE.pdb       1ERE_A_fixed.pdb     EST_redock_1ERE_A.sdf  
1GWR  1GWR.pdb       1GWR_A_fixed.pdb     EST_redock_1GWR_A.sdf  
3UUD  3UUD.pdb       3UUD_A_fixed.pdb     EST_redock_3UUD_A.sdf  
6CBZ  6CBZ.pdb       6CBZ_A_fixed.pdb     EST_redock_6CBZ_A.sdf  
3ERD  3ERD.pdb       3ERD_A_fixed.pdb     DES_redock_3ERD_A.sdf  
4ZN7  4ZN7.pdb       4ZN7_A_fixed.pdb     DES_redock_4ZN7_A.sdf  
4MGC  4MGC.pdb       4MGC_A_fixed.pdb     27M_redock_4MGC_A.sdf  
4MG8  4MG8.pdb       4MG8_A_fixed.pdb     27J_redock_4MG8_A.sdf  
4TUZ  4TUZ.pdb       4TUZ_A_fixed.pdb     36J_redock_4TUZ_A.sdf  
3UU7  3UU7.pdb       3UU7_A_fixed.pdb     2OH_redock_3UU7_A.sdf  
4MG9  4MG9.pdb       4MG9_A_fixed.pdb     27K_redock_4MG9_A.sdf  
4MGA  4MGA.pdb       4MGA_A_fixed.pdb     27L_redock_4MGA_A.sdf  
1G50  1G50.pdb       1G50_A_fixed.pdb     EST_redock_1G50_A.sdf  


## Populate ideal ligand dictionary

In [ ]:
# Obsolete - This block assumes that these ligands are already present in the 'minimized_ideal_ligands' directory.
#minimized versions of ideal ligands

idealliganddict = {}

idealliganddict["27J_min"] = {'id':"27J_min", 'prep': "obabel-mmff94", 'localfilename': "27J_min.sdf"}
idealliganddict["27K_min"] = {'id':"27K_min", 'prep': "obabel-mmff94", 'localfilename': "27K_min.sdf"}
idealliganddict["27L_min"] = {'id':"27L_min", 'prep': "obabel-mmff94", 'localfilename': "27L_min.sdf"}

idealliganddict["27M_min"] = {'id':"27M_min", 'prep': "obabel-mmff94", 'localfilename': "27M_min.sdf"}
idealliganddict["2OH_min"] = {'id':"2OH_min", 'prep': "obabel-mmff94", 'localfilename': "2OH_min.sdf"}
idealliganddict["36J_min"] = {'id':"36J_min", 'prep': "obabel-mmff94", 'localfilename': "36J_min.sdf"}

idealliganddict["Caffeine_min"] = {'id':"Caffeine_min", 'prep': "obabel-mmff94", 'localfilename': "Caffeine_min.sdf"}
idealliganddict["DES_min"] = {'id':"DES_min", 'prep': "obabel-mmff94", 'localfilename': "DES_min.sdf"}
idealliganddict["EE2_min"] = {'id':"EE2_min", 'prep': "obabel-mmff94", 'localfilename': "EE2_min.sdf"}

idealliganddict["EST_min"] = {'id':"EST_min", 'prep': "obabel-mmff94", 'localfilename': "EST_min.sdf"}
idealliganddict["Melatonin_min"] = {'id':"Melatonin_min", 'prep': "obabel-mmff94", 'localfilename': "Melatonin_min.sdf"}
idealliganddict["Testosterone_min"] = {'id':"Testosterone_min", 'prep': "obabel-mmff94", 'localfilename': "Testosterone_min.sdf"}

### Add pubchem sourced ligands (optional)

In [15]:
# This block adds ideal ligands to the dictionary that were sourced from PubChem via CAS number by the 'acquire-ligand-cas-minimize' notebook.
# These files need to be present in the "ligand/[project]/processed" folder and start with 'cas' in the filename.

import os
import re

idealliganddict = {}

for entry in os.scandir(sourceliganddir):
    if entry.is_file() and entry.name.endswith(".sdf"):
        print ("I see file " + entry.name)
        id = "test" + entry.name
        match = re.search(r'^cas-(.*)\.sdf$', entry.name)
        if (match):
            id = match.group(1)       
            idealliganddict[id] = {'id':id, 'prep': "obabel-mmff94", 'localfilename': entry.name}
            print("     I added file " + entry.name)
            

I see file cas-58-73-1_min.sdf
     I added file cas-58-73-1_min.sdf
I see file cas-134523-00-5_min.sdf
     I added file cas-134523-00-5_min.sdf
I see file cas-298-46-4_min.sdf
     I added file cas-298-46-4_min.sdf
I see file cas-93413-69-5_min.sdf
     I added file cas-93413-69-5_min.sdf
I see file cas-85-68-7_min.sdf
     I added file cas-85-68-7_min.sdf
I see file cas-58-08-2_min.sdf
     I added file cas-58-08-2_min.sdf
I see file cas-134-62-3_min.sdf
     I added file cas-134-62-3_min.sdf


In [16]:
lines = reportdict(idealliganddict, ["id","prep","localfilename"])
print("\n".join(lines))
print("Number of ligands: " + str(len(idealliganddict)))

id               prep           localfilename          
58-73-1_min      obabel-mmff94  cas-58-73-1_min.sdf    
134523-00-5_min  obabel-mmff94  cas-134523-00-5_min.sdf
298-46-4_min     obabel-mmff94  cas-298-46-4_min.sdf   
93413-69-5_min   obabel-mmff94  cas-93413-69-5_min.sdf 
85-68-7_min      obabel-mmff94  cas-85-68-7_min.sdf    
58-08-2_min      obabel-mmff94  cas-58-08-2_min.sdf    
134-62-3_min     obabel-mmff94  cas-134-62-3_min.sdf   
Number of ligands: 7


## Examine proteins to determine what chains and ligands are present in the proteins. (Optional informational step) ##

In [ ]:
import gemmi

for key, value in proteindict.items():
    
    structure = gemmi.read_structure(prodir + value['localfilename_fixed'])
    #structure = gemmi.read_structure(chemfilesdir + "6O4w_rcbs.pdb")
    ligands = []
    
    for model in structure:
        for chain in model:
            for res in chain:
                if res.het_flag != ' ':  # hetero-residue
                    if res.name not in ("HOH", "WAT", "H2O"):
                        #if res.seqid.num == 604:
                        ligands.append((res.name, chain.name, res.seqid.num))

    print("protein: " + value['localfilename'])
    print(set(ligands))

In [ ]:
from Bio.PDB import PDBParser

for key, value in proteindict.items():
    
  parser = PDBParser(QUIET=True)
  structure = parser.get_structure("prot", prodir + value['localfilename_fixed'])
  #structure = parser.get_structure("prot", prodir + value['localfilename_fixed'])
    
  print("Protein: " + value['localfilename'])
    
  for model in structure:
    print(f"  Model {model.id}:")
    chain_ids = [chain.id for chain in model]
    print("    Chains:", ", ".join(chain_ids))


## Visualize Ligands

In [ ]:
from rdkit import Chem

ligandfile = ligdir + '2R6_ideal_PubChem.sdf'
ligandfileout = ligdir + '2R6_ideal_PubChem_NOH.sdf'

# Load SDF file (remove Hs on read - most efficient)
mol = Chem.MolFromMolFile(ligandfile, removeHs=True)

# Or if already loaded with Hs:
# mol = Chem.MolFromMolFile("ligand.sdf", removeHs=False)
# mol = Chem.RemoveHs(mol)

# Write H-free SDF
writer = Chem.SDWriter(ligandfileout)
writer.write(mol)
writer.close()

print(f"Atoms before: {Chem.MolFromMolFile(ligandfileout, removeHs=False).GetNumAtoms()}")
print(f"Atoms after:  {mol.GetNumAtoms()}")

In [ ]:
import nglview as nv
from rdkit import Chem

# From SDF
view = nv.show_structure_file(ligdir + '2R6_ideal_PubChem.sdf')
view.add_representation('ball+stick')
view.camera = 'orthographic'
view.center()
view

In [ ]:
view = nv.show_structure_file(ligdir + '4o09_final_ligand_2R6_A.pdb')
view.add_representation('ball+stick')
view.camera = 'orthographic'
view.center()
view

In [ ]:
view = nv.show_structure_file(ligdir + '2R6_redock_4o09_final_A_obabel.sdf')
view.add_representation('ball+stick')
view.camera = 'orthographic'
view.center()
view

In [ ]:
view.display(gui=True) 

## Define Gnina and dataframe functions

In [52]:
#Define the functions that call Gnina, parse the results files, and write those results to the dataframe.
from rdkit import Chem
import subprocess
from pathlib import Path

number_of_modes = 5 #Report how many modes from each Gnina run?

def executegnina(proteinid,ligandid):
    protein = proteindict[proteinid]
    ligand = idealliganddict[ligandid]
    #box = dockedliganddict[boxid]
    p = sourceproteindir + protein["localfilename_fixed"]
    l = sourceliganddir + ligand["localfilename"]
    #b = dockeddir + box["localfilename"]
    b = referenceliganddir + protein["ligandreferencefilename"]
    return callgnina(p,l,b)


def executegninadecoy(proteinid,boxid):
    protein = proteindict[proteinid]
    box = dockedliganddict[boxid]
    p = prodir + protein["localfilename_fixed"]
    l = decoydir + "decoyligand.sdf"
    b = dockeddir + box["localfilename"]
    return callgnina(p,l,b)

    
def callgnina(p,l,b):

    Path(docked).write_text("") #truncate Gnina output file before writing a new output file.
    
    #!~/octoberproject/gnina -r "{p}" -l "{l}" --autobox_ligand "{b}" -o "{docked}" --log "{log}" --exhaustiveness=16 --num_modes=8 --seed 0 --pose_sort_order CNNaffinity --no_gpu  
    #!~/octoberproject/gnina -r "{p}" -l "{l}" --autobox_ligand "{b}" -o "{docked}" --log "{log}" --exhaustiveness=16 --num_modes=9 --seed 0 --pose_sort_order CNNscore --no_gpu  
    #!"{GNINA_LOC}" -r "{p}" -l "{l}" --autobox_ligand "{b}" -o "{docked}" --log "{log}" --exhaustiveness=16 --num_modes=9 --seed 0 --pose_sort_order CNNscore "{GNINA_PARAMETER}"

    if not GNINA_PARAMETER:
        gnina_cmd = [GNINA_LOC, "-r", p, "-l", l, "--autobox_ligand", b, "--autobox_add", "4", "-o", docked, "--log", log, 
       "--exhaustiveness=16", "--num_modes=9", "--seed", "0", "--pose_sort_order", "CNNscore"]

    else:
        gnina_cmd = [GNINA_LOC, "-r", p, "-l", l, "--autobox_ligand", b, "--autobox_add", "4", "-o", docked, "--log", log, 
        "--exhaustiveness=16", "--num_modes=9", "--seed", "0", "--pose_sort_order", "CNNscore", GNINA_PARAMETER]

        #gnina_cmd = [GNINA_LOC, "-r", p, "-l", l, "--autobox_ligand", b, "--autobox_add", "4", "-o", docked, "--log", log, 
        #"--exhaustiveness=16", "--num_modes=9", "--seed", "0", "--pose_sort_order", "CNNscore", 
        #"--scoring", "vinardo", "--cnn_scoring", "rescore", GNINA_PARAMETER]

        #gnina_cmd = [GNINA_LOC, "-r", p, "-l", l, "--autobox_ligand", b, "--autobox_add", "4", "-o", docked, "--log", log, 
        #"--exhaustiveness=16", "--num_modes=9", "--seed", "0", "--pose_sort_order", "CNNscore", 
        #"--scoring", "vinardo", "--cnn_scoring", "none", GNINA_PARAMETER]

    #return True #To skip the call for Gnina 
    
    print("Call gnina:", " ".join(gnina_cmd))
 
    try:
        result = subprocess.run(gnina_cmd, check=True, capture_output=True, text=True)
        print("stdout:", result.stdout)
    except subprocess.CalledProcessError as e:
        # FAILURE (returncode != 0)
        print("Failure")
        print(f"Return code: {e.returncode}")
        print("stderr:", e.stderr)
        return False
    else:
        #only execute obrms if gnina ran successfully
        #Compute RMSD of two docked ligand files and save results to rmsdlog file.
        !obrms -f "{b}" "{docked}" | tee "{rmsdlog}"
        return True
    
    
def getdockedresultdf(docked):
    rmsddf = pd.read_csv(rmsdlog, sep=" ", header=None)
    rows = []
    for i, mol in enumerate(Chem.SDMolSupplier(docked)):
        if mol is None:
            continue
        rows.append({
            "pose": i,
            "CNNscore": float(mol.GetProp("CNNscore")),
            "CNN_VS": float(mol.GetProp("CNN_VS")),
            "CNNaffinity": float(mol.GetProp("CNNaffinity")),
            "RMSD": float(rmsddf.iloc[i,2]),
            "affinity": float(mol.GetProp("minimizedAffinity")),
        })
    df = pd.DataFrame(rows)
    if(sort_by_cnnvs):
        df.sort_values(by="CNN_VS", ascending=False, inplace=True)
    return(df)

def writeresulttodf(pro,lig,box):
    resultsdf = getdockedresultdf(docked)
    #for index, row in resultsdf.iterrows():
    for index, row in resultsdf.head(number_of_modes).iterrows():
        #write to the resultsdf here.
        temp = [pro, lig, box, "{:.4f}".format(row['CNNscore']), "{:.4f}".format(row['CNN_VS']), "{:.4f}".format(row['RMSD']), "{:.4f}".format(row['affinity'])]
        outputdf.loc[len(outputdf)] = temp
        #write to rawresults.csv here.
        pd.DataFrame([temp]).to_csv(rawresults, mode="a", header=False, index=False)
        


In [53]:
resultsdf = getdockedresultdf(docked)
print("resultsdf:")
print(resultsdf)

resultsdf:
   pose  CNNscore    CNN_VS  CNNaffinity  RMSD  affinity
0     0  0.212397  1.532583     7.215655   inf  11.82337
1     1  0.185798  1.325810     7.135741   inf   3.16990
2     2  0.170745  0.983328     5.759045   inf   6.11664
3     3  0.160267  1.074919     6.707058   inf   4.52084
4     4  0.157393  0.934118     5.934953   inf   5.21431
5     5  0.131553  0.733768     5.577713   inf   6.08881
6     6  0.130617  0.943949     7.226875   inf   5.92794
7     7  0.129867  0.764924     5.890059   inf   4.85633
8     8  0.124446  0.659504     5.299536   inf   5.85968


[10:42:50] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[10:42:50] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[10:42:50] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[10:42:50] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[10:42:50] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[10:42:50] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[10:42:50] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[10:42:50] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[10:42:50] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol

## Run the docking simulations

In [59]:
# Establish new empty dataframe
# Or set it to empty for the next new run.

import pandas as pd
from pathlib import Path

outputdf = pd.DataFrame(columns = ["protein", "ideal_ligand", "native_ligand", "CNN_pose", "CNN_VS", "RMSD", "affinity"])
Path(rawresults).write_text("") #truncate raw results file for each new run.
header = ["protein", "ideal_ligand", "native_ligand", "CNN_pose", "CNN_VS", "RMSD", "affinity"]
pd.DataFrame([header]).to_csv(rawresults, mode="a", header=False, index=False)

In [55]:
#Test block. Dock just one pair.
pkey = "1ERE"
ikey = "58-73-1_min"
dockedid = "EST_redock_1ERE" # This variable used only for results file. The function executegnina() looks up the reference ligand from protein dict.
#gninasuccess = executegnina(pkey, ikey, dockedid)
gninasuccess = executegnina(pkey, ikey)
if (gninasuccess):
    writeresulttodf(pkey, ikey, dockedid) 

Call gnina: /home/dwaine/octoberproject/gnina -r protein/protein-er-13/processed/1ERE_A_fixed.pdb -l ligand/erligand260/processed/cas-58-73-1_min.sdf --autobox_ligand protein/protein-er-13/reference_ligand/EST_redock_1ERE_A.sdf --autobox_add 4 -o results/docked.sdf --log results/gninalog.txt --exhaustiveness=16 --num_modes=9 --seed 0 --pose_sort_order CNNscore --no_gpu
stdout:               _             
             (_)            
   __ _ _ __  _ _ __   __ _ 
  / _` | '_ \| | '_ \ / _` |
 | (_| | | | | | | | | (_| |
  \__, |_| |_|_|_| |_|\__,_|
   __/ |                    
  |___/                     

gnina v1.3.2 master:f23dd2b   Built Jul 29 2025.
gnina is based on smina and AutoDock Vina.
Please cite appropriately.

Commandline: /home/dwaine/octoberproject/gnina -r protein/protein-er-13/processed/1ERE_A_fixed.pdb -l ligand/erligand260/processed/cas-58-73-1_min.sdf --autobox_ligand protein/protein-er-13/reference_ligand/EST_redock_1ERE_A.sdf --autobox_add 4 -o results/docked.sdf 

[10:43:37] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[10:43:37] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[10:43:37] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[10:43:37] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[10:43:37] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[10:43:37] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[10:43:37] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[10:43:37] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[10:43:37] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol

In [60]:
# Iterate though protein dictionary and ligand dictionary and perform Gnina docking for every combo.
from itertools import islice

# For testing a subset of proteins or ligands set these variables. Set to zero for all ligands or proteins.
number_of_proteins = 2
number_of_liands = 2

pro = proteindict
if (number_of_proteins > 0) :
  pro = dict(islice(proteindict.items(), number_of_proteins))
ide = idealliganddict
if (number_of_liands > 0) :
  ide = dict(islice(idealliganddict.items(), number_of_liands))

rundecoys = False

start = time.time()

#iterate through proteins
for pkey, pvalue in pro.items():
    #iterate through ideal ligands
    for ikey, ivalue in ide.items():
        print(f"{pkey} meets {ikey} at {pvalue['ligandreferencefilename']}")
        executegnina(pkey, ikey)
        writeresulttodf(pkey,ikey,pvalue['ligandreferencefilename'])   

    #iterate through decoy ligands if dataframe exists and is populated
    if (rundecoys):
        for idx, row in decoydf.head(2).iterrows():
        #for idx, row in decoydf.iterrows():
            writeonedecoytoFS(row)
            print(f"{pkey} meets {row['ID']} at {pvalue['dockedid']}") 
            executegninadecoy(pkey, pvalue['dockedid'])
            writeresulttodf(pkey, row['ID'], pvalue['dockedid'])  

end = time.time()
elapsed = end - start
print(f"Gnina runs completed in {format_time(elapsed)}")  # 00:01:05

1ERE meets 58-73-1_min at EST_redock_1ERE_A.sdf
Call gnina: /home/dwaine/octoberproject/gnina -r protein/protein-er-13/processed/1ERE_A_fixed.pdb -l ligand/erligand260/processed/cas-58-73-1_min.sdf --autobox_ligand protein/protein-er-13/reference_ligand/EST_redock_1ERE_A.sdf --autobox_add 4 -o results/docked.sdf --log results/gninalog.txt --exhaustiveness=16 --num_modes=9 --seed 0 --pose_sort_order CNNscore --no_gpu
stdout:               _             
             (_)            
   __ _ _ __  _ _ __   __ _ 
  / _` | '_ \| | '_ \ / _` |
 | (_| | | | | | | | | (_| |
  \__, |_| |_|_|_| |_|\__,_|
   __/ |                    
  |___/                     

gnina v1.3.2 master:f23dd2b   Built Jul 29 2025.
gnina is based on smina and AutoDock Vina.
Please cite appropriately.

Commandline: /home/dwaine/octoberproject/gnina -r protein/protein-er-13/processed/1ERE_A_fixed.pdb -l ligand/erligand260/processed/cas-58-73-1_min.sdf --autobox_ligand protein/protein-er-13/reference_ligand/EST_redock_1

[10:49:28] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[10:49:28] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[10:49:28] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[10:49:28] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[10:49:28] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[10:49:28] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[10:49:28] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[10:49:28] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[10:49:28] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol

stdout:               _             
             (_)            
   __ _ _ __  _ _ __   __ _ 
  / _` | '_ \| | '_ \ / _` |
 | (_| | | | | | | | | (_| |
  \__, |_| |_|_|_| |_|\__,_|
   __/ |                    
  |___/                     

gnina v1.3.2 master:f23dd2b   Built Jul 29 2025.
gnina is based on smina and AutoDock Vina.
Please cite appropriately.

Commandline: /home/dwaine/octoberproject/gnina -r protein/protein-er-13/processed/1ERE_A_fixed.pdb -l ligand/erligand260/processed/cas-134523-00-5_min.sdf --autobox_ligand protein/protein-er-13/reference_ligand/EST_redock_1ERE_A.sdf --autobox_add 4 -o results/docked.sdf --log results/gninalog.txt --exhaustiveness=16 --num_modes=9 --seed 0 --pose_sort_order CNNscore --no_gpu
Using random seed: 0

0%   10   20   30   40   50   60   70   80   90   100%
|----|----|----|----|----|----|----|----|----|----|
***************************************************
60823 | pose 0 | initial pose not within box
60823 | pose 0 | ligand outside box


[10:51:14] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[10:51:14] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[10:51:14] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[10:51:14] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[10:51:14] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[10:51:14] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[10:51:14] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[10:51:14] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[10:51:14] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol

stdout:               _             
             (_)            
   __ _ _ __  _ _ __   __ _ 
  / _` | '_ \| | '_ \ / _` |
 | (_| | | | | | | | | (_| |
  \__, |_| |_|_|_| |_|\__,_|
   __/ |                    
  |___/                     

gnina v1.3.2 master:f23dd2b   Built Jul 29 2025.
gnina is based on smina and AutoDock Vina.
Please cite appropriately.

Commandline: /home/dwaine/octoberproject/gnina -r protein/protein-er-13/processed/1GWR_A_fixed.pdb -l ligand/erligand260/processed/cas-58-73-1_min.sdf --autobox_ligand protein/protein-er-13/reference_ligand/EST_redock_1GWR_A.sdf --autobox_add 4 -o results/docked.sdf --log results/gninalog.txt --exhaustiveness=16 --num_modes=9 --seed 0 --pose_sort_order CNNscore --no_gpu
Using random seed: 0

0%   10   20   30   40   50   60   70   80   90   100%
|----|----|----|----|----|----|----|----|----|----|
***************************************************
3100 | pose 0 | initial pose not within box

mode |  affinity  |  intramol  |    CNN 

[10:51:42] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[10:51:42] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[10:51:42] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[10:51:42] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[10:51:42] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[10:51:42] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[10:51:42] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[10:51:42] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[10:51:42] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol

stdout:               _             
             (_)            
   __ _ _ __  _ _ __   __ _ 
  / _` | '_ \| | '_ \ / _` |
 | (_| | | | | | | | | (_| |
  \__, |_| |_|_|_| |_|\__,_|
   __/ |                    
  |___/                     

gnina v1.3.2 master:f23dd2b   Built Jul 29 2025.
gnina is based on smina and AutoDock Vina.
Please cite appropriately.

Commandline: /home/dwaine/octoberproject/gnina -r protein/protein-er-13/processed/1GWR_A_fixed.pdb -l ligand/erligand260/processed/cas-134523-00-5_min.sdf --autobox_ligand protein/protein-er-13/reference_ligand/EST_redock_1GWR_A.sdf --autobox_add 4 -o results/docked.sdf --log results/gninalog.txt --exhaustiveness=16 --num_modes=9 --seed 0 --pose_sort_order CNNscore --no_gpu
Using random seed: 0

0%   10   20   30   40   50   60   70   80   90   100%
|----|----|----|----|----|----|----|----|----|----|
***************************************************
60823 | pose 0 | initial pose not within box

mode |  affinity  |  intramol  |   

[10:54:02] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[10:54:02] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[10:54:02] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[10:54:02] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[10:54:02] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[10:54:02] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[10:54:02] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[10:54:02] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[10:54:02] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol

In [ ]:
print(f"Gnina runs completed in {format_time(elapsed)}")  # 00:01:05

In [61]:
# Preview results
print(outputdf)

   protein     ideal_ligand          native_ligand CNN_pose  CNN_VS RMSD  \
0     1ERE      58-73-1_min  EST_redock_1ERE_A.sdf   0.5159  2.9968  inf   
1     1ERE      58-73-1_min  EST_redock_1ERE_A.sdf   0.4669  2.6816  inf   
2     1ERE      58-73-1_min  EST_redock_1ERE_A.sdf   0.4530  2.6304  inf   
3     1ERE      58-73-1_min  EST_redock_1ERE_A.sdf   0.4242  2.4080  inf   
4     1ERE      58-73-1_min  EST_redock_1ERE_A.sdf   0.3888  2.1501  inf   
5     1ERE  134523-00-5_min  EST_redock_1ERE_A.sdf   0.2047  1.4940  inf   
6     1ERE  134523-00-5_min  EST_redock_1ERE_A.sdf   0.1663  1.2520  inf   
7     1ERE  134523-00-5_min  EST_redock_1ERE_A.sdf   0.1631  0.9521  inf   
8     1ERE  134523-00-5_min  EST_redock_1ERE_A.sdf   0.1540  0.8618  inf   
9     1ERE  134523-00-5_min  EST_redock_1ERE_A.sdf   0.1498  1.0738  inf   
10    1GWR      58-73-1_min  EST_redock_1GWR_A.sdf   0.6290  3.5408  inf   
11    1GWR      58-73-1_min  EST_redock_1GWR_A.sdf   0.6140  3.6099  inf   
12    1GWR  

## Order, rearrance, output as .csv

In [30]:
# Order by LIGAND, then CNN_pose desc. Change order of columns.
outputdfsorted = pd.DataFrame(columns = ["protein", "ideal_ligand", "native_ligand", "CNN_pose", "CNN_VS", "RMSD", "affinity"])
for group_name, group_df in outputdf.groupby("ideal_ligand"):
    oneliganddf = group_df.sort_values(by="CNN_pose", ascending=False)
    outputdfsorted = pd.concat([outputdfsorted, pd.DataFrame(oneliganddf)], ignore_index=True)
    
outputdfsorted = outputdfsorted[["ideal_ligand", "protein", "native_ligand", "CNN_pose", "CNN_VS", "RMSD", "affinity"]]

print(outputdfsorted)

  ideal_ligand protein    native_ligand CNN_pose  CNN_VS RMSD affinity
0  58-73-1_min    1ERE  EST_redock_1ERE   0.5159  2.9968  inf  -7.5032
1  58-73-1_min    1ERE  EST_redock_1ERE   0.4669  2.6816  inf  -7.9233
2  58-73-1_min    1ERE  EST_redock_1ERE   0.4530  2.6304  inf  -7.0437
3  58-73-1_min    1ERE  EST_redock_1ERE   0.4242  2.4080  inf  -7.3705
4  58-73-1_min    1ERE  EST_redock_1ERE   0.3888  2.1501  inf  -7.5092


In [62]:
# Order by PROTEIN, then CNN_pose desc. Change order of columns.
outputdfsorted = pd.DataFrame(columns = ["protein", "ideal_ligand", "native_ligand", "CNN_pose", "CNN_VS", "RMSD", "affinity"])
for group_name, group_df in outputdf.groupby("protein"):
    oneliganddf = group_df.sort_values(by="CNN_pose", ascending=False)
    outputdfsorted = pd.concat([outputdfsorted, pd.DataFrame(oneliganddf)], ignore_index=True)
    
outputdfsorted = outputdfsorted[["protein", "ideal_ligand", "native_ligand", "CNN_pose", "CNN_VS", "RMSD", "affinity"]]

print(outputdfsorted)

   protein     ideal_ligand          native_ligand CNN_pose  CNN_VS RMSD  \
0     1ERE      58-73-1_min  EST_redock_1ERE_A.sdf   0.5159  2.9968  inf   
1     1ERE      58-73-1_min  EST_redock_1ERE_A.sdf   0.4669  2.6816  inf   
2     1ERE      58-73-1_min  EST_redock_1ERE_A.sdf   0.4530  2.6304  inf   
3     1ERE      58-73-1_min  EST_redock_1ERE_A.sdf   0.4242  2.4080  inf   
4     1ERE      58-73-1_min  EST_redock_1ERE_A.sdf   0.3888  2.1501  inf   
5     1ERE  134523-00-5_min  EST_redock_1ERE_A.sdf   0.2047  1.4940  inf   
6     1ERE  134523-00-5_min  EST_redock_1ERE_A.sdf   0.1663  1.2520  inf   
7     1ERE  134523-00-5_min  EST_redock_1ERE_A.sdf   0.1631  0.9521  inf   
8     1ERE  134523-00-5_min  EST_redock_1ERE_A.sdf   0.1540  0.8618  inf   
9     1ERE  134523-00-5_min  EST_redock_1ERE_A.sdf   0.1498  1.0738  inf   
10    1GWR      58-73-1_min  EST_redock_1GWR_A.sdf   0.6290  3.5408  inf   
11    1GWR      58-73-1_min  EST_redock_1GWR_A.sdf   0.6140  3.6099  inf   
12    1GWR  

In [63]:
# Write to .csv
import csv
import time
epoch = int(time.time())
outfile = resultsdir + "results" + str(epoch) + ".csv"
outputdfsorted.to_csv(outfile, mode='w', index=False, header=True)